# Hyperparameter search with Optuna

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/LitWBHSTrainingBasicNeuralNetwork/lit_wbhs_training_basic_neural_network_notebook.ipynb)

This notebook searches the hyperparameters of the fully connected network of `LitWBHSTrainingBasicNeuralNetwork` with [Optuna](https://optuna.org), then trains the best settings found. The final training run is logged to Weights & Biases (W&B); the search trials are not, unless you turn it on (section 4).

### Choose your route

| Route | Sections to run | Search and train with |
| --- | --- | --- |
| **Colab, from notebook cells** | 1 (setup), then 2 to 5 | The cells of sections 3 and 5 |
| **Colab, from Colab's terminal** (Colab Pro) | 1 (setup) in the notebook first; 2, 4, and 5 to read and use the results | The terminal commands in sections 3 and 5; the reload cell then loads the study into the notebook |
| **Locally, from this notebook** | Skip 1, start at [**2. The search space**](#2.-The-search-space) | The cells of sections 3 and 5, with the `.venv` as the notebook's kernel |
| **Locally, from your terminal** | None, no notebook needed (optionally 2, 4, and 5 for the plots) | `python -m LitWBHSTrainingBasicNeuralNetwork.search --config search01.json`, then `python -m LitWBHSTrainingBasicNeuralNetwork --config search01_best.json`, from the repo root with the `.venv` activated |
| **Colab CLI, for the final training run** | None, no notebook needed | `python -m LitWBHSTrainingBasicNeuralNetwork.search --config search01.json`, then `python -m LitWBHSTrainingBasicNeuralNetwork --config search01_best.json` on a Colab runtime opened from your terminal: see [**Training with the Colab CLI**](#Training-with-the-Colab-CLI) |

In Colab, the setup cells are needed even when you search in the terminal: Google Drive can only be mounted, and Colab Secrets only read, from a notebook cell. Locally, set up the repo once first (`.venv`, requirements, and `.env`, see the main README), and open this notebook from the project folder.

In Colab, run the cells from top to bottom:

1. **Set up** the runtime (Colab only): GPU, Google Drive, the repo, and your W&B key.
2. **Look at the search space**: what is searched, and in which ranges.
3. **Run the search**: Optuna trains many settings and keeps the best one.
4. **Read the results**: the best trial and Optuna's plots.
5. **Train the best config** and test it.

## 1. Setup (Colab)

> **Running locally?** Skip this section: these cells only work in Colab. Jump to [**2. The search space**](#2.-The-search-space).

The same setup as in the other projects' notebooks. First choose a GPU: *Runtime → Change runtime type → GPU*. Then run these cells once per session (and again after the runtime resets): mount Drive, set up the repo, change directory, and load API keys.

Results, including the Optuna study, go to Google Drive, so a search interrupted by a disconnect can be continued (see the tips at the end).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e

cd /content
if [ -d CodeCS4337Fall2026/.git ]; then
  git -C CodeCS4337Fall2026 pull --ff-only
else
  git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
fi
cd CodeCS4337Fall2026

# Colab settings: data on the local disk, results in Google Drive.
cp .envcolab .env

# Install only packages Colab does not already have (versions are not pinned).
grep -vE '^\s*(#|$)' requirements.txt \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then
        echo "skip $pkg (already installed)"
      else
        pip install -q "$pkg" && echo "installed $pkg"
      fi
    done

In [ ]:
# The bash cell above cannot change the notebook's working directory, so do it here.
%cd /content/CodeCS4337Fall2026

### Load your W&B key (optional)

Add your key as a Colab **Secret** named `WANDB_API_KEY` (key icon in the left sidebar → *Add new secret*). Never type it into a cell. The cell below loads `.env`, then looks up missing keys in your Secrets, adds them to this notebook's environment, and saves them to `.env` so terminal runs find them too. Without a key, the training run of section 5 is logged to W&B offline and can be uploaded later with `wandb sync`.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.colab import userdata

ENV_FILE = Path("/content/CodeCS4337Fall2026/.env")
KEYS = ["WANDB_API_KEY"]

if load_dotenv(ENV_FILE):
    print(f"Loaded settings from {ENV_FILE}")
else:
    print(f"No settings in {ENV_FILE}: run the setup cells first")

for key in KEYS:
    if os.getenv(key):
        print(f"{key}: found in .env")
        continue
    try:
        os.environ[key] = userdata.get(key)
    except Exception:  # noqa: BLE001 (Secret not added or access not granted)
        print(f"{key}: not found, W&B will log offline")
        continue
    with ENV_FILE.open("a") as env_file:
        env_file.write(f"\n{key}={os.environ[key]}\n")
    print(f"{key}: loaded from Colab Secrets and saved to .env")

## 2. The search space

This cell makes the project importable (in Colab and locally) and imports its two entry points: `search` runs the Optuna study, `main` trains and tests one config.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = Path("/content/CodeCS4337Fall2026") if IN_COLAB else Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from LitWBHSTrainingBasicNeuralNetwork import main
from LitWBHSTrainingBasicNeuralNetwork.search import search
from LitWBHSTrainingBasicNeuralNetwork.utils import CONFIGS_DIR

print((CONFIGS_DIR / "search01.json").read_text())

### Where your results go

Run the cell below and look at what it prints: `OUTPUT_DIR` is the folder where **every run of this project is saved**: checkpoints, metrics, plots, and `runs_summary.csv`. It is set in `.env` at the repo root.

- **In Colab** it is in your Google Drive, `/content/drive/MyDrive/CodeCS4337Fall2026/runs` (from `.envcolab`), so your results survive runtime resets. Open *MyDrive → CodeCS4337Fall2026 → runs* in Google Drive to find them later.
- **Locally** it is `runs`, i.e. the `runs/` folder of the repo (relative paths are relative to the repo root).

Each search gets one folder, `<OUTPUT_DIR>/LitWBHSTrainingBasicNeuralNetwork/<search name>/` (the study, `trials.csv`, the best config), and each training run its own, `<OUTPUT_DIR>/LitWBHSTrainingBasicNeuralNetwork/<config>/<timestamp>/`.

In [ ]:
import os

from dotenv import load_dotenv

# Colab: the copy of .envcolab made by the setup cell. Locally: your own .env.
load_dotenv(REPO / ".env")
!echo "OUTPUT_DIR=$OUTPUT_DIR"
output_dir = REPO / os.getenv("OUTPUT_DIR", "runs")
print(f"Runs of this project are saved in: {output_dir / 'LitWBHSTrainingBasicNeuralNetwork'}")

The search config has three parts:

- `base_config`: the training config every trial starts from. Everything not searched (seed, validation split, W&B project) comes from it.
- `study`: how many trials to run (`n_trials`), an optional time limit (`timeout_minutes`), the sampler's seed, the fraction of the training data each trial uses (`train_fraction`), the pruner settings, and whether trials are logged to W&B.
- `search_space`: a range `[low, high]` or a list of choices for every hyperparameter.

| Hyperparameter | What it controls |
| --- | --- |
| `n_layers`, `units` | Number of hidden layers, and the units of each one (picked per layer) |
| `activation` | ReLU, Leaky ReLU, GELU, or tanh after every hidden layer |
| `dropout` | Dropout probability after every hidden layer |
| `batch_size` | Samples per training step |
| `epochs` | Maximum number of epochs |
| `optimizer`, `lr`, `momentum` | Adam, AdamW, SGD, or RMSprop; the learning rate (searched on a log scale); momentum (SGD and RMSprop only) |
| `regularization`, `l1`, `l2` | No penalty, L1 (pushes weights to zero), L2 (weight decay, shrinks weights), or both, with their weights (log scale) |
| `scheduler`, `step_size`, `gamma` | Constant learning rate, step decay, cosine decay, or reduce-on-plateau, with their settings |
| `early_stopping`, `patience` | Whether to stop when validation accuracy stops improving, and after how many epochs |

Some hyperparameters only exist for some choices, e.g. `momentum` only for SGD and RMSprop. Optuna handles this naturally: the search space is defined by running Python code (`suggest_config` in `search.py`), so an `if` decides what is sampled.

**To change the search**, copy `search01.json` in the file browser (left sidebar, `CodeCS4337Fall2026/LitWBHSTrainingBasicNeuralNetwork/configs/`), e.g. to `search02.json`, edit it, and pass its name below. To fix a hyperparameter instead of searching it, give a single choice (`["adam"]`) or a range with equal ends (`[10, 10]`). Fewer searched hyperparameters means fewer trials are needed.

## 3. Run the search

Each trial trains one setting on the training split and is scored by its best **validation** accuracy. The test set is never used during the search: choosing settings by their test score would make that score optimistic.

- Optuna's **TPE sampler** picks each trial's settings from how earlier trials did: after the first random trials, it samples more where the scores were good.
- The **median pruner** stops a trial early when its validation accuracy after an epoch is below the median of earlier trials at the same epoch. Pruned trials save time for promising ones.

A trial takes from a few seconds to a minute or more on a GPU, so 30 trials take roughly 10 to 30 minutes. Try `N_TRIALS = 3` first to check that everything works. Running the cell again **adds** trials to the same study instead of starting over.

**Faster trials with less data:** `TRAIN_FRACTION` is the fraction of the 54,000 training images each trial trains on. In Colab, which has only 2 CPU cores to load images, `0.25` makes every epoch about 4 times faster. The validation set stays complete, so the scores remain reliable, and the best config is still trained on all the data in section 5. The trade-off: settings that depend on the amount of data (epochs, dropout, L1/L2 strength) may not transfer exactly, so prefer `1.0` when you have the time. Keep the same fraction for all trials of a study; to change it, use a new search config name.

In [ ]:
import warnings

# Harmless warnings printed for every trial: Lightning still uses a PyTorch class
# that is being renamed, and Python warns about the DataLoader starting workers.
warnings.filterwarnings("ignore", message=r".*isinstance\(treespec, LeafSpec\).*")
warnings.filterwarnings("ignore", message=r".*use of fork\(\) may lead to deadlocks.*")

SEARCH_CONFIG = "search01.json"
N_TRIALS = 50
TRAIN_FRACTION = 0.25  # e.g. 0.25 in Colab for faster trials

study = search(SEARCH_CONFIG, n_trials=N_TRIALS, train_fraction=TRAIN_FRACTION)

**Or from a terminal** (the same search; the notebook stays free while it runs in Colab's terminal):

```bash
cd /content/CodeCS4337Fall2026
python -m LitWBHSTrainingBasicNeuralNetwork.search --config search01.json --n-trials 30 --train-fraction 1.0
```

Then load the study in the notebook with the next cell (`N_TRIALS = 0` runs no new trial).

In [ ]:
# Reload a saved study (after a terminal search or a runtime reset) without new trials.
study = search(SEARCH_CONFIG, n_trials=0)

## 4. Read the results

The ten best trials, best first:

In [ ]:
from html import escape

from IPython.display import HTML, display
from optuna.trial import TrialState

complete = study.get_trials(states=[TrialState.COMPLETE])
pruned = study.get_trials(states=[TrialState.PRUNED])
print(f"{len(complete)} complete, {len(pruned)} pruned")

top = sorted(complete, key=lambda t: t.value, reverse=True)[:10]
# Columns in the order the settings are picked; "-" where a trial has no such setting.
params = list(dict.fromkeys(name for t in top for name in t.params))


def fmt(value):
    return f"{value:.4g}" if isinstance(value, float) else escape(str(value))


header = "".join(f"<th>{escape(c)}</th>" for c in ["trial", "val_acc", "epochs_run", *params])
rows = "".join(
    f"<tr><td>{t.number}</td><td><b>{t.value:.4f}</b></td><td>{len(t.intermediate_values)}</td>"
    + "".join(f"<td>{fmt(t.params[p]) if p in t.params else '-'}</td>" for p in params)
    + "</tr>"
    for t in top
)
display(
    HTML(
        '<div style="overflow-x: auto; max-width: 100%;">'
        '<table style="white-space: nowrap; border-collapse: collapse;">'
        f"<thead><tr>{header}</tr></thead><tbody>{rows}</tbody></table></div>"
    )
)

### Optuna's plots

- **Optimization history:** each trial's score, and the best so far. It should rise quickly, then flatten: more trials then bring little.
- **Intermediate values:** the validation accuracy of every trial, epoch by epoch. Pruned trials are the short lines that stop early.
- **Hyperparameter importances:** how much of the variation in score each hyperparameter explains. Important ones deserve a finer search; unimportant ones can be fixed.
- **Slice plot:** the score against the value of one hyperparameter, one dot per trial. Where are the good learning rates?

In [ ]:
import warnings

import matplotlib.pyplot as plt
import optuna
from optuna.visualization import matplotlib as optuna_plots

# Optuna's matplotlib plots are marked experimental; the warning is not useful here.
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

optuna_plots.plot_optimization_history(study)
plt.show()
optuna_plots.plot_intermediate_values(study)
plt.show()
optuna_plots.plot_param_importances(study)
plt.show()
optuna_plots.plot_slice(study, params=["lr", "dropout", "optimizer", "n_layers"])
plt.show()

In [ ]:
optuna_plots.plot_timeline(study)
optuna_plots.plot_contour(study, params=["lr", "dropout"])
optuna_plots.plot_rank(study, params=["lr", "dropout", "batch_size"])
optuna_plots.plot_edf(study)
optuna_plots.plot_terminator_improvement(study)

### Optional: the trials in W&B

By default the trials are not logged to W&B: one W&B run per trial makes every trial slower and fills the project with dozens of runs, and the plots above already compare them. To log them, set `"log_trials_to_wandb": true` in the search config's `"study"` before searching. Every trial is then a W&B run in the project `LitWBHSTrainingBasicNeuralNetwork`, grouped under the search's name (`search01`), with its settings in the run config and `best_val_acc` and `state` (`complete` or `pruned`) in the summary. On wandb.ai:

1. In the **Runs** table, group by `group`, show the columns you care about (`training.optimizer`, `training.lr`, `best_val_acc`, ...), and sort by `best_val_acc`.
2. Add a **Parallel coordinates** panel with the searched settings and `best_val_acc`: one line per trial, so you can follow the best lines back to their settings.
3. Add a **Parameter importance** panel for `best_val_acc`: W&B's own estimate of which settings matter.
4. The `val_acc` and `lr-...` charts overlay all trials, showing what the schedulers and early stopping did.

Without a key, upload the trials later with the `wandb sync` command printed during the search.

## 5. Train the best config

The search saved the best trial's settings as a normal training config, `configs/search01_best.json` (and a copy, `best_config.json`, next to the study in Drive). Its `found_by` entry records which trial it came from.

In [ ]:
best_config_name = f"{Path(SEARCH_CONFIG).stem}_best.json"
print((CONFIGS_DIR / best_config_name).read_text())

Train it like any other config: a full run with checkpoints, then **one** evaluation on the test set, which is the score to report. The run is logged to W&B in the group `search01_best`, next to the trials. The same from a terminal: `python -m LitWBHSTrainingBasicNeuralNetwork --config search01_best.json`.

In [ ]:
best_metrics = main(best_config_name)
print(f"Test accuracy of the best config: {best_metrics['test_acc']:.4f}")

**Compare with the starting point** (optional): `config01.json` holds the default settings (Adam, learning rate 0.001, two hidden layers of 256 and 128 units, 10 epochs). Did the search beat it on the test set?

In [ ]:
baseline_metrics = main("config01.json")
print(
    f"Baseline {baseline_metrics['test_acc']:.4f}, best config {best_metrics['test_acc']:.4f}"
)

## Tips

**Long searches in Colab**
- The study is saved in Google Drive (`MyDrive/CodeCS4337Fall2026/runs/LitWBHSTrainingBasicNeuralNetwork/search01/study.db`) after every trial. After a disconnect, re-run the setup cells and the search cell: the finished trials are kept and new ones are added.
- `configs/search01_best.json` is in `/content`, which is deleted when the runtime resets. Get it back with the reload cell (`n_trials=0`), which rewrites it from the study.
- Set `timeout_minutes` in the search config to stop starting new trials after a time limit.
- The [pro tips in the main README](https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026#pro-tips-keep-training-running) (idle disconnects, GPU quota) apply here too.

**Searching well**
- Start wide, then narrow: after a first search, read the importances and slice plots, fix the unimportant hyperparameters, and search the important ones in a smaller range (`search02.json`).
- To start a search over, use a new search config name, or delete its folder in Drive.
- Fewer epochs per trial make the search faster, but favor settings that learn fast rather than settings that end best.

## Training with the Colab CLI

Another way to train this project, once everything is final. Its only purpose
is the training run itself:

1. **Work locally** on the project: change the code or a config, run short
   tests, fix bugs.
2. **Use Google Colab** (this notebook) for debugging and interactive work.
3. **Use the Colab CLI** when everything is final and the only thing left is
   training. The CLI creates a Colab runtime from your local terminal and opens
   a shell on it, so there is no notebook or browser tab to keep open. On the
   runtime you clone the repo and run `python -m LitWBHSTrainingBasicNeuralNetwork.search --config search01.json`.

The Colab CLI works on Linux and macOS (not Windows). For more detail, see
[google-colab-cli.md](https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/google-colab-cli.md) at the repo root or the
[official Colab CLI documentation](https://github.com/googlecolab/google-colab-cli).

### Step 1: Install the CLI

On your own machine:

```bash
uv tool install google-colab-cli     # or: pip install google-colab-cli
colab version                         # check that it works
```

The first command that talks to Colab prints a sign-in link: open it, sign in
with your Google account, and paste the code it shows back into the terminal.

### Step 2: The basic workflow (sessions)

A **session** is one Colab runtime (a virtual machine) with a name you choose,
here `cs4337`. These commands run in your **local** terminal:

| What | Command |
| --- | --- |
| Create a session with a GPU | `colab new -s cs4337 --gpu T4` (or `L4`, `A100`, `H100`, depending on your plan) |
| Create a session without a GPU | `colab new -s cs4337` (CPU only) |
| List your sessions / check one | `colab sessions` / `colab status -s cs4337` |
| Attach to a session (open a shell on it) | `colab ssh -s cs4337` |
| Leave the shell; the session keeps running | `exit` |
| Stop the session (delete the runtime) | `colab stop -s cs4337` |

Use a GPU runtime: `--gpu T4` is enough for this project. A CPU session can't be switched to a GPU: stop it and create a new one.

> **Once a session is stopped, nothing on it stays.** The cloned repo, the
> downloaded data, and every file that is not on Google Drive are deleted.
> This project writes its runs to Google Drive (step 4), so mount Drive before
> training.

### Step 3: Mount Google Drive

Drive is mounted from your local terminal, not from inside the session. If you
are attached, `exit` first:

```bash
exit                                # only if you are inside the session
colab drivemount -s cs4337          # open the link it prints and allow access
colab ssh -s cs4337                 # attach again
ls /content/drive/MyDrive           # check: your Drive files are listed
```

### Step 4: Clone the repo and train

Inside the session, clone the repo and go to its root folder:

```bash
cd /content
git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
cd CodeCS4337Fall2026
cp .envcolab .env
```

The runtime only sees what is on GitHub: if you changed code or a config
locally, push it to your own fork first and clone that instead. `.envcolab`
sets `DATA_DIR=/content/data` (data on the runtime's fast local disk) and
`OUTPUT_DIR=/content/drive/MyDrive/CodeCS4337Fall2026/runs` (runs on Drive, so they are kept after a stop).

This project logs to Weights & Biases and reads the key only from `.env`.
Add your W&B API key (from [wandb.ai/authorize](https://wandb.ai/authorize))
to the runtime's `.env`. `read -s` hides the key while you paste it, so it is
not shown on screen or saved in the shell history:

```bash
read -rsp "W&B API key: " key && echo "WANDB_API_KEY=$key" >> .env && unset key && echo
```

Without a key the run still trains, but W&B logs offline into the run folder;
upload it later with the `wandb sync` command that the run prints. Never put
the key in `.envcolab`, which is in git.

Install only the packages Colab doesn't already have, as the setup of this notebook
does. A plain `pip install -r requirements.txt` would replace Colab's own
versions, which is slow and can break other preinstalled packages:

```bash
grep -vE '^\s*(#|$)' requirements.txt \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then echo "skip $pkg (already installed)"
      else pip install -q "$pkg" && echo "installed $pkg"; fi
    done
```

Start a `tmux` session first, so the search keeps going if the SSH connection
drops or you close your laptop. Then, from the repo root, run the search and
train the best config it finds, one after the other:

```bash
tmux new -s train                  # if tmux is missing: apt-get install -y tmux
python -m LitWBHSTrainingBasicNeuralNetwork.search --config search01.json \
  && python -m LitWBHSTrainingBasicNeuralNetwork --config search01_best.json
```

The search saves the best config as `configs/search01_best.json` inside the
cloned repo, which is deleted when the session stops. Its study (`study.db`)
and `trials.csv` are on Drive, though: if the session stopped before the final
training, clone again and run
`python -m LitWBHSTrainingBasicNeuralNetwork.search --config search01.json --n-trials 0` to write
`search01_best.json` back without running any new trial. Running the same
search command again (without `--n-trials 0`) continues the saved study.

To leave training running, press `Ctrl+B`, then `D`. To come back later:
`colab ssh -s cs4337`, then `tmux attach -t train`.

### Step 5: Check that everything was logged

The run folders are on Google Drive, so they are kept after the session
stops. From the session:

```bash
ls /content/drive/MyDrive/CodeCS4337Fall2026/runs/LitWBHSTrainingBasicNeuralNetwork/search01_best
```

or on [drive.google.com](https://drive.google.com), under **My Drive >
CodeCS4337Fall2026 > runs > LitWBHSTrainingBasicNeuralNetwork**. Check that:

- each run has its own `<timestamp>` folder (the path is printed when the run starts), with its config, results, plots, and checkpoints;
- `runs_summary.csv`, next to the run folders, has one new row per run;
- `/content/drive/MyDrive/CodeCS4337Fall2026/runs/LitWBHSTrainingBasicNeuralNetwork/search01/` has the search's `trials.csv` and `study.db`;
- the run shows up in your W&B project: the run prints its W&B link. If it printed a `wandb sync` command instead, the run was logged offline (no key or no login); run that command later, from a machine with your key, to upload it;
- if the session stopped or the run was cut off, start a new session, repeat the steps above, and continue from the last checkpoint on Drive with `--resume-from search01_best`, e.g. `python -m LitWBHSTrainingBasicNeuralNetwork --config search01_best.json --resume-from search01_best`.

When everything is checked, stop the session so it no longer uses your Colab
quota:

```bash
colab stop -s cs4337
```